# Asistente Inteligente (RAG)
Este notebook usará los chunks almacenados para brindarle contexto al LLM, el cual evaluará y dará recomendaciones sobre los requerimientos que escribamos.

In [1]:
!source env/bin/activate
%pip install langchain langchain-community langchain-google-genai psycopg2-binary pgvector tiktoken dotenv

Note: you may need to restart the kernel to use updated packages.


## Configurar variables de entorno y LLM

In [2]:
import os
from dotenv import load_dotenv
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores.pgvector import PGVector

load_dotenv()

db_name = os.getenv("POSTGRES_DB")
db_password = os.getenv("POSTGRES_PASSWORD")
db_user = os.getenv("POSTGRES_USER")
db_host = os.getenv("POSTGRES_HOST", "localhost")
db_port = os.getenv("POSTGRES_PORT", "5432")
gemini_api_key = os.getenv("GOOGLE_API_KEY", "key not found")

CONNECTION_STRING = f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
COLLECTION_NAME = "iso_standards"

# componentes principales
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

# Temperatura baja para que sea conciso y certero
llm = ChatGoogleGenerativeAI(
    model="models/gemini-2.5-flash",
    temperature=0.1,
    gemini_api_key = gemini_api_key
)

# vectorial db conection
vector_store = PGVector(
    connection_string=CONNECTION_STRING,
    embedding_function=embeddings,
    collection_name=COLLECTION_NAME
)


Unexpected argument 'gemini_api_key' provided to ChatGoogleGenerativeAI. Did you mean: 'google_api_key'?
/home/xlancet/Codes/Pasantia/implementation/NLP4RE/env/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3747: UserWarning: WARNING! gemini_api_key is not default parameter.
                gemini_api_key was transferred to model_kwargs.
                Please confirm that gemini_api_key is what you intended.
  exec(code_obj, self.user_global_ns, self.user_ns)
/tmp/ipykernel_69595/4113796826.py:29: LangChainPendingDeprecationWarning: This class is pending deprecation and may be removed in a future version. You can swap to using the `PGVector` implementation in `langchain_postgres`. Please read the guidelines in the doc-string of this class to follow prior to migrating as there are some differences between the implementations. See <https://github.com/langchain-ai/langchain-postgres> for details about the new implementation.
  vector_store = PGVector(
/tmp/ipykernel_69595/

## Definir el Prompt para el Evaluador

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

import json
with open('./docs/rules.json', 'r', encoding='utf-8') as f: rules_data = json.load(f)
with open('./docs/examples.json', 'r', encoding='utf-8') as f: examples_data = json.load(f)

rules_str = ""
for r in rules_data:
    rules_str += f"- {r['typeC']}: {r['description']}\n"
    for level, desc in r['criterion'].items(): rules_str += f"  * Nivel {level}: {desc}\n"

examples_str = ""
for idx, ex in enumerate(examples_data):
    examples_str += f"Ejemplo {idx+1}:\nRequerimiento: {ex['text']}\nEvaluación:\n"
    for tag, score in ex['tags'].items(): examples_str += f" - {tag}: {score} ({ex['explanations'].get(tag, 'OK')})\n"
    examples_str += "\n"

template_rag = f"""Eres un experto en Ingeniería de Requisitos en una pasantía de investigación.
Se te proporcionará contexto normativo basado en la ISO 29148.
Tu tarea es recibir un requerimiento de un usuario, evaluarlo y sugerir cómo mejorarlo basándote en las siguientes reglas y ejemplos.

Reglas de evaluación:
{rules_str}

Ejemplos de cómo evaluar:
{examples_str}

Contexto normativo recuperado de la ISO:
{{context}}

Requerimiento propuesto por el usuario:
{{requirement}}

Instrucciones:
1. Analiza el requerimiento del usuario basándote en las métricas y reglas (VERIFIABILITY, ATOMICITY, AMBIGUITY, COMPLETENESS, TRACEABLE).
2. Da recomendaciones específicas para mejorarlo.
3. Presenta una versión reescrita y óptima del requerimiento."""

prompt_rag = ChatPromptTemplate.from_template(template_rag)

## Cadena RAG y Pruebas

In [4]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# configurar la cadena para que recupere los k trozos con mayor similitud a la consulta
k = 3

retriever = vector_store.as_retriever(search_kwargs={"k": k})

In [ ]:
print(retriever)

tags=['PGVector', 'GoogleGenerativeAIEmbeddings'] vectorstore=<langchain_community.vectorstores.pgvector.PGVector object at 0x7df1b38ffb00> search_kwargs={'k': 3}


In [ ]:


chain_rag = (
    {"context": retriever, "requirement": RunnablePassthrough()}
    | prompt_rag
    | llm
    | StrOutputParser()
)

# hacer la prueba con un requerimiento deficiente e impreciso
requerimiento_usuario = "El sistema de base de datos deberá ser muy rápido y soportar bastantes cosas sin caerse."

print("Evaluando requerimiento...\n")
resultado = chain_rag.invoke(requerimiento_usuario)
print(resultado)

Evaluando requerimiento...

¡Excelente! Como experto en Ingeniería de Requisitos en esta pasantía de investigación, procederé a evaluar el requerimiento propuesto por el usuario basándome en las reglas y ejemplos proporcionados, y en el contexto normativo de la ISO 29148.

---

**Requerimiento propuesto por el usuario:**
"El sistema de base de datos deberá ser muy rápido y soportar bastantes cosas sin caerse."

---

**Evaluación exhaustiva:**

*   **VERIFIABILITY:** 0
    *   Los términos "muy rápido" y "bastantes cosas sin caerse" son completamente subjetivos y carecen de cualquier métrica o condición medible. No hay una forma objetiva de probar si el sistema cumple con estos criterios. La ISO 29148 enfatiza que los requisitos de rendimiento deben ser "declarados en términos medibles" (ISO 29148, p. 79).
*   **ATOMICITY:** 0
    *   El requerimiento expresa dos ideas independientes unidas por la conjunción "y":
        1.  "El sistema de base de datos deberá ser muy rápido" (relaciona